# Analizimi i Çmimeve të Produkteve në Big Data
### Deep Learning — Metodat dhe Aplikimi

## 1. Simulimi i të Dhënave

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime
import os

fake = Faker()
np.random.seed(42)
random.seed(42)

FOLDER = 'C:/Users/COMPUTER SHOP/OneDrive/Desktop/FikrijeIdrizi_DeepLearning'

print('Libraritë u importuan me sukses!')

Libraritë u importuan me sukses!


In [2]:
# Kategorite dhe brendet
categories = {
    'Electronics': ['Samsung', 'Apple', 'Sony', 'LG', 'Huawei'],
    'Clothing':    ['Zara', 'H&M', 'Nike', 'Adidas', 'Puma'],
    'Food':        ['Nestle', 'Unilever', 'Kraft', 'Pepsi', 'Coca-Cola'],
    'Sports':      ['Nike', 'Adidas', 'Puma', 'Under Armour', 'Reebok'],
    'Home':        ['IKEA', 'Philips', 'Bosch', 'Braun', 'Electrolux']
}

base_prices = {
    'Electronics': (200, 1500),
    'Clothing':    (20,  200),
    'Food':        (2,   50),
    'Sports':      (30,  300),
    'Home':        (15,  500)
}

print('Kategorite:', list(categories.keys()))
print('Çmimet bazë:', base_prices)

Kategorite: ['Electronics', 'Clothing', 'Food', 'Sports', 'Home']
Çmimet bazë: {'Electronics': (200, 1500), 'Clothing': (20, 200), 'Food': (2, 50), 'Sports': (30, 300), 'Home': (15, 500)}


In [3]:
# Gjenerimi i 50 produkteve
n_products = 50
products = []
for i in range(1, n_products + 1):
    category = random.choice(list(categories.keys()))
    brand    = random.choice(categories[category])
    pmin, pmax = base_prices[category]
    base_price = round(random.uniform(pmin, pmax), 2)
    products.append({'product_id': f'P{i:03d}', 'category': category,
                     'brand': brand, 'base_price': base_price})

print(f'{len(products)} produkte u gjeneruan!')
pd.DataFrame(products).head(10)

50 produkte u gjeneruan!


,product_id,category,brand,base_price
0,P001,Electronics,Samsung,1164.02
1,P002,Clothing,H&M,45.12
2,P003,Electronics,Huawei,313.02
3,P004,Sports,Nike,38.05
4,P005,Clothing,H&M,110.96
5,P006,Electronics,Huawei,458.49
6,P007,Home,Braun,121.91
7,P008,Home,Bosch,407.57
8,P009,Electronics,Apple,1107.58
9,P010,Food,Kraft,9.46


In [4]:
# Gjenerimi i rekordeve ditore (50 produkte x 365 ditë)
def get_season(date):
    m = date.month
    if m in [12,1,2]: return 'Winter'
    if m in [3,4,5]:  return 'Spring'
    if m in [6,7,8]:  return 'Summer'
    return 'Fall'

dates   = pd.date_range(start='2024-01-01', end='2024-12-31', freq='D')
records = []

for product in products:
    prev_price = product['base_price']
    for date in dates:
        season        = get_season(date)
        season_factor = {'Winter':1.10,'Spring':0.95,'Summer':1.05,'Fall':1.00}[season]
        promotion_flag = 1 if ((date.weekday()>=5 and random.random()<0.30) or random.random()<0.05) else 0
        discount_rate  = round(random.uniform(0.05,0.30) if promotion_flag else random.uniform(0.00,0.08), 3)
        noise          = np.random.normal(0, product['base_price']*0.02)
        own_price      = round(max(product['base_price']*season_factor*(1-discount_rate)+noise, 1.0), 2)
        competitor_price = round(own_price*random.uniform(0.85,1.15), 2)
        if own_price < competitor_price and promotion_flag: demand_level = 'High'
        elif own_price > competitor_price*1.10:             demand_level = 'Low'
        else:                                               demand_level = random.choice(['Low','Medium','High'])
        dm          = {'Low':0.3,'Medium':0.6,'High':1.0}[demand_level]
        stock_level = max(0, int(random.uniform(50,1000)*(1-dm*0.5)))
        price_change_rate = round((own_price-prev_price)/prev_price*100, 4) if prev_price else 0.0
        prev_price  = own_price
        records.append({'product_id':product['product_id'],'date':date.strftime('%Y-%m-%d'),
                        'category':product['category'],'brand':product['brand'],
                        'own_price':own_price,'competitor_price':competitor_price,
                        'demand_level':demand_level,'stock_level':stock_level,
                        'discount_rate':discount_rate,'season':season,
                        'promotion_flag':promotion_flag,'price_change_rate':price_change_rate})

df_raw = pd.DataFrame(records)
print(f'Rekorde totale : {len(df_raw):,}')
print(f'Produkte       : {df_raw["product_id"].nunique()}')
print(f'Periudha       : {df_raw["date"].min()} -> {df_raw["date"].max()}')
df_raw.head()

Rekorde totale : 18,300
Produkte       : 50
Periudha       : 2024-01-01 -> 2024-12-31


,product_id,date,category,brand,own_price,competitor_price,demand_level,stock_level,discount_rate,season,promotion_flag,price_change_rate
0,P001,2024-01-01,Electronics,Samsung,1259.98,1092.88,Low,751,0.025,Winter,0,8.2438
1,P001,2024-01-02,Electronics,Samsung,1268.24,1262.91,Low,832,0.007,Winter,0,0.6556
2,P001,2024-01-03,Electronics,Samsung,1282.70,1344.17,High,103,0.010,Winter,0,1.1402
3,P001,2024-01-04,Electronics,Samsung,1253.14,1427.71,High,383,0.049,Winter,0,-2.3045
4,P001,2024-01-05,Electronics,Samsung,1201.99,1165.57,High,333,0.057,Winter,0,-4.0817


In [5]:
# Ruajtja e të dhënave
df_raw.to_csv(f'{FOLDER}/price_data.csv', index=False)
print(f'U ruajt: price_data.csv ({len(df_raw):,} rekorde)')
print('Shperndarje sipas kategorise:')
print(df_raw['category'].value_counts())

U ruajt: price_data.csv (18,300 rekorde)
Shperndarje sipas kategorise:
category
Electronics    4392
Home           4392
Clothing       3294
Food           3294
Sports         2928
Name: count, dtype: int64


## 2. Importimi në MongoDB

In [8]:
from pymongo import MongoClient, errors
import pandas as pd

try:
    FOLDER
except NameError:
    FOLDER = 'C:/Users/COMPUTER SHOP/OneDrive/Desktop/FikrijeIdrizi_DeepLearning'

client = MongoClient('mongodb://localhost:27017/', serverSelectionTimeoutMS=5000)
db     = client['price_db']

mongo_available = True
try:
    client.admin.command('ping')
except errors.ServerSelectionTimeoutError as e:
    print('MongoDB nuk është në dispozicion në localhost:27017.')
    print(f'  {e}')
    print('Shikoni nëse MongoDB është i ndezur dhe ri-provo.')
    mongo_available = False

if mongo_available:
    df_csv = pd.read_csv(f'{FOLDER}/price_data.csv')
    print(f'Database : price_db | Host: localhost:27017')
    print(f'Rekorde  : {len(df_csv):,}')
else:
    df_csv = None

Database : price_db | Host: localhost:27017
Rekorde  : 18,300


In [ ]:
if mongo_available:
    # Importimi ne 3 koleksione
    db['price_history'].drop()
    db['products'].drop()
    db['market_summary'].drop()

    # Koleksioni 1: price_history
    result = db['price_history'].insert_many(df_csv.to_dict(orient='records'))
    print(f'[1] price_history : {len(result.inserted_ids):,} dokumente')

    # Koleksioni 2: products
    prods = df_csv[['product_id','category','brand']].drop_duplicates().to_dict(orient='records')
    db['products'].insert_many(prods)
    print(f'[2] products      : {len(prods)} dokumente')

    # Koleksioni 3: market_summary
    df_csv['date']  = pd.to_datetime(df_csv['date'])
    df_csv['month'] = df_csv['date'].dt.to_period('M').astype(str)
    market = df_csv.groupby(['product_id','month']).agg(
        avg_own_price=('own_price','mean'), avg_competitor_price=('competitor_price','mean'),
        promotions_count=('promotion_flag','sum'), avg_price_change=('price_change_rate','mean')
    ).round(3).reset_index()
    db['market_summary'].insert_many(market.to_dict(orient='records'))
    print(f'[3] market_summary: {len(market)} dokumente')

    # Indekse
    db['price_history'].create_index('product_id')
    db['price_history'].create_index('date')
    print('\nMongoDB u ngarkua me sukses!')
else:
    print('MongoDB importi u anulua sepse shërbimi nuk është i disponueshëm.')

[1] price_history : 18,300 dokumente
[2] products      : 50 dokumente
[3] market_summary: 600 dokumente

MongoDB u ngarkua me sukses!


In [10]:
# Verifikimi
print('Verifikimi i koleksioneve:')
print(f'  price_history  : {db["price_history"].count_documents({}):,} dok.')
print(f'  products       : {db["products"].count_documents({}):,} dok.')
print(f'  market_summary : {db["market_summary"].count_documents({}):,} dok.')
print()
print('Shembull rekord:')
sample = db['price_history'].find_one({'product_id':'P001'}, {'_id':0})
for k,v in sample.items():
    print(f'  {k:<22}: {v}')

Verifikimi i koleksioneve:
  price_history  : 18,300 dok.
  products       : 50 dok.
  market_summary : 600 dok.

Shembull rekord:
  product_id            : P001
  date                  : 2024-01-01
  category              : Electronics
  brand                 : Samsung
  own_price             : 1259.98
  competitor_price      : 1092.88
  demand_level          : Low
  stock_level           : 751
  discount_rate         : 0.025
  season                : Winter
  promotion_flag        : 0
  price_change_rate     : 8.2438


## 3. Pastrimi & Përpunimi i të Dhënave

In [11]:
# Ngarkimi nga MongoDB
df = pd.DataFrame(list(db['price_history'].find({}, {'_id':0})))
df['date']           = pd.to_datetime(df['date'])
df['promotion_flag'] = df['promotion_flag'].astype(int)
df['stock_level']    = df['stock_level'].astype(int)
print(f'Te dhena te ngarkuara: {len(df):,} rekorde')

Te dhena te ngarkuara: 18,300 rekorde


In [12]:
# Kontrolli i cilesise se te dhenave
print('[1] Vlera null:')
nulls = df.isnull().sum()
print('   ', nulls[nulls>0].to_dict() if nulls.sum()>0 else 'Asnje null -- OK')

print(f'\n[2] Dublikate: {df.duplicated(subset=["product_id","date"]).sum()}')

print('\n[3] Outliers (IQR):')
for col in ['own_price','competitor_price','price_change_rate']:
    Q1,Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    out   = ((df[col]<Q1-1.5*(Q3-Q1))|(df[col]>Q3+1.5*(Q3-Q1))).sum()
    print(f'   {col:<25}: {out} outliers')

[1] Vlera null:
    Asnje null -- OK

[2] Dublikate: 0

[3] Outliers (IQR):
   own_price                : 2587 outliers
   competitor_price         : 2388 outliers
   price_change_rate        : 1999 outliers


In [13]:
# Feature Engineering
df = df.sort_values(['product_id','date'])
df['day_of_week']      = df['date'].dt.dayofweek
df['month']            = df['date'].dt.month
df['day_of_year']      = df['date'].dt.dayofyear
df['is_weekend']       = (df['date'].dt.dayofweek>=5).astype(int)
df['price_diff']       = df['own_price'] - df['competitor_price']
df['price_diff_pct']   = ((df['own_price']-df['competitor_price'])/df['competitor_price']*100).round(4)
df['effective_price']  = (df['own_price']*(1-df['discount_rate'])).round(2)
df['price_ma7']        = df.groupby('product_id')['own_price'].transform(lambda x: x.rolling(7,min_periods=1).mean()).round(2)
df['price_ma30']       = df.groupby('product_id')['own_price'].transform(lambda x: x.rolling(30,min_periods=1).mean()).round(2)
df['price_volatility'] = df.groupby('product_id')['own_price'].transform(lambda x: x.rolling(7,min_periods=1).std()).round(4).fillna(0)
df['demand_encoded']   = df['demand_level'].map({'Low':0,'Medium':1,'High':2})
df['season_encoded']   = df['season'].map({'Winter':0,'Spring':1,'Summer':2,'Fall':3})

print(f'Dataset final: {len(df):,} rekorde x {len(df.columns)} variabla')
df[['own_price','competitor_price','effective_price','price_volatility']].describe().round(2)

Dataset final: 18,300 rekorde x 24 variabla


,own_price,competitor_price,effective_price,price_volatility
count,18300.00,18300.00,18300.00,18300.00
mean,357.75,358.07,338.49,20.19
std,401.35,404.78,382.72,27.22
min,4.35,4.03,3.09,0.00
25%,47.41,48.94,45.36,2.70
50%,174.48,177.92,167.06,9.57
75%,435.91,441.25,418.11,27.48
max,1515.04,1653.99,1513.52,203.06


In [14]:
# Ruajtja e dataset-it te pastruar
df.to_csv(f'{FOLDER}/price_data_clean.csv', index=False)
db['price_clean'].drop()
db['price_clean'].insert_many(df.to_dict(orient='records'))
print(f'U ruajt: price_data_clean.csv ({len(df):,} rekorde, {len(df.columns)} kolona)')
print('MongoDB: price_clean u perditesua!')
client.close()

U ruajt: price_data_clean.csv (18,300 rekorde, 24 kolona)
MongoDB: price_clean u perditesua!
